In [1]:
import pandas as pd

df = pd.read_csv("moto_imu_sim/imu_data.csv")
print(df.dtypes, df.isna().sum())
g = df.groupby("ride_id")
print(g.size().describe())
print(g['label'].nunique().value_counts())

ride_id        str
t          float64
ax         float64
ay         float64
az         float64
gx         float64
gy         float64
gz         float64
label          str
dtype: object ride_id    0
t          0
ax         0
ay         0
az         0
gx         0
gy         0
gz         0
label      0
dtype: int64
count       36.000000
mean     59805.805556
std         26.020672
min      59762.000000
25%      59786.000000
50%      59810.000000
75%      59823.000000
max      59868.000000
dtype: float64
label
1    36
Name: count, dtype: int64


STEP 2: SAMPLING REGULARITY :Checking whether the time distribution, or the time between reading is constant

In [2]:
dt = df.groupby("ride_id")["t"].diff().dropna()# groupby ride_id, but subtract time"t" consecutively i.e row 2 - row 1, row 3 - row 2 and so on. so is basically time difference dt
print(dt.describe())
print(dt.round(6).value_counts().head())
print("non-positive:", (dt <= 0).sum(), "| gaps >7.5ms:", (dt > 0.0075).sum())

count    2.152973e+06
mean     5.016235e-03
std      5.787643e-04
min      1.493000e-03
25%      4.857000e-03
50%      5.001000e-03
75%      5.144000e-03
max      5.543100e-02
Name: t, dtype: float64
t
0.005019    4188
0.004964    4165
0.004981    4142
0.005011    4119
0.005021    4114
Name: count, dtype: int64
non-positive: 0 | gaps >7.5ms: 5232


In [3]:
df["dt"] = df.groupby("ride_id")["t"].diff()
df["gap"] = df["dt"] > 0.0075 # df["gap"] keeps the boolean value of dt col, if it's greater than 0.0075

In [4]:
print(df.groupby("label")["gap"].mean())

label
aggressive    0.002523
calm          0.002373
normal        0.002427
Name: gap, dtype: float64


essentially similar


STEP 3: CLIPPING AND VALUE RANGES

Reason: sensor have a max range (eg: +/- 16g). Anything beyond will be cutoff flat and values gather at exactly the max. Hides the agressiveness of the ride 

In [5]:
print(df[["ax", "ay", "az", "gx", "gy", "gz"]].agg(["min", "max"]))

           ax        ay        az          gx          gy         gz
min -1.380615 -1.071289 -2.388428 -112.884521 -108.123779 -53.344727
max  1.444336  1.078613  5.138184  114.471436   93.475342  50.140381


aggegerated the min and max of each col.

no sign of clipping neither min nor max saturate at the sensor max value


STEP 4: GRAVITY AXIS PER RIDE.(mount leak check)

checking how the sensor was mounted in the bike
and which axis the gravity is acting


In [6]:
m = df.groupby("ride_id")[["ax", "ay", "az"]].mean() # mean forces, ie cornering and braking cancel over time but gravity stays consistent and survives the mean
m["label"] = df.groupby("ride_id")["label"].first() #since each ride has same label taking first one is fine
print(m.sort_values("label"))

               ax        ay        az       label
ride_id                                          
R002    -0.009381 -0.019232  1.047785  aggressive
R004    -0.048712  0.045065  1.090882  aggressive
R005    -0.098032 -0.062199  0.999784  aggressive
R006     0.011512  0.000320  1.113729  aggressive
R007    -0.006551  0.007377  1.031298  aggressive
R032    -0.021686  0.016803  0.993761  aggressive
R031     0.018692  0.114509  1.080766  aggressive
R020    -0.086482  0.016941  1.039926  aggressive
R001    -0.041379 -0.011114  0.961665        calm
R033    -0.097293 -0.049270  0.989642        calm
R029     0.075334 -0.024977  1.057383        calm
R027    -0.059902 -0.015717  0.988272        calm
R021    -0.049948  0.094717  0.942698        calm
R019    -0.066460  0.043149  0.993900        calm
R035    -0.015543 -0.042931  0.992679        calm
R018     0.027275  0.001886  1.001748        calm
R014    -0.011002  0.024721  0.995267        calm
R013     0.007046  0.059542  1.001943        calm


STEP 5: PER-RIDE SPREAD(standard deviation) BY LABEL 
  

In [7]:
s =  df.groupby('ride_id')[["ax", "ay", "az", "gx", "gy", "gz"]].std()
s["label"] = m["label"]
print(s.groupby("label").mean())

                  ax        ay        az         gx        gy         gz
label                                                                   
aggressive  0.261784  0.086920  0.273299  10.678578  9.616056  11.426227
calm        0.114804  0.038976  0.188802   4.751677  6.214186   9.281170
normal      0.170689  0.051730  0.182902   6.449247  6.288456   9.858385


here groupedby ride_id (has around 36 rides) , and find their standard deviation 
then label is reused form m
last print line groups he above 36 rows with label and find the mean standard deviation


Checking overlap between rides in each label

In [8]:
print(s.groupby("label")["gx"].describe())

            count       mean       std       min       25%        50%  \
label                                                                   
aggressive    8.0  10.678578  2.534997  6.765928  9.592610  10.993182   
calm         12.0   4.751677  0.951400  3.078163  4.253321   4.706801   
normal       16.0   6.449247  1.236646  4.279310  5.651999   6.364073   

                  75%        max  
label                             
aggressive  12.092548  14.158948  
calm         5.406144   6.180134  
normal       6.889196   9.486926  


here, ranges:
Calm: 3.08 to 6.18
Normal: 4.28 to 9.49
Aggressive: 6.77 to 14.16
here calm and agressive are barely seperated but normal is heavily overlapped with both of the other labels, so model would struggle with this.

to remember we are evaluating the deviation(std()) not the raw values, just the deviation of the values from mean